# Project Part 1: Data Pipeline — Cleaning & Sentiment Analysis

**Run this notebook at least once before starting Part 2 (Whitepaper). If you find that your map needs additional corrections, you will need to run Part 3 and 4 again.**

This notebook takes your school's raw geoparsed data, applies your team's location review, runs sentiment analysis, and exports files that every subsequent notebook depends on. Each processed dataset is saved in two formats: `.csv` for inspection and `.pickle` to preserve dtypes for Plotly maps.

| Output files | Used by |
|---|---|
| `data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned.csv` / `.pickle` | Part 2 (Whitepaper) before/after map |
| `data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned_sentiment.csv` / `.pickle` | Part 2 (Whitepaper) sentiment maps, Part 3 Base Map, Part 4 (Flythrough) |

The notebook ends with a git branch → commit → pull request workflow to submit your cleaned data for grading.

## Overview

- **Prerequisites:** Your school's geoparsed data exists in `data/{SCHOOL}/` (generated by the instructor)
- **Parts:**
  1. Load raw geoparsed data
  2. Review locations in Google Sheets
  3. Load and validate the review sheet from Google Sheets
  4. Apply corrections and export the cleaned data file
  5. Run RoBERTa sentiment analysis and export results
  6. Commit and push to GitHub

## 1 Load Raw Geoparsed Data

Fill in `SCHOOL` below and run the cell. You will fill in `SHEETS_LINK` later, after completing the Google Sheets review in section 2.

1. **`SCHOOL`** — your assigned school abbreviation. Must match the folder name exactly.
   - Valid values: `GMU`, `ODU`, `UNC`, `UVA`, `VCU`, `VirginiaTech`, `WM`
2. **`SHEETS_LINK`** — leave this as the placeholder for now. After completing section 2, paste your published CSV URL here and re-run this cell.

> 👉 **Note:** *Every cell in this notebook uses `SCHOOL` automatically — you only need to set it once here.*

The raw geoparsed file was created by the instructor. It contains every sentence from your school's subreddit that mentioned a place name, with the geoparser's best guess at coordinates. 

In [ ]:
import pandas as pd

SCHOOL     = "UNC"                      # ← replace: GMU, ODU, UNC, UVA, VCU, VirginiaTech, WM
SHEETS_LINK = "https://docs.google.com/spreadsheets/d/e/2PACX-1vTWWrPkOD7taFxFfd4o1a06zAhBXpNzMWhLoYDoQx4Q0WDuWWC305-HBTAhoLI-7XuPuYP_e9dPOmHQ/pub?output=csv"  # ← fill in after completing section 2

# ── Derived paths (do not change) ─────────────────────────────────────────
RAW_PATH       = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_long.csv'
CLEANED_PATH   = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned'
SENTIMENT_PATH = f'../data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned_sentiment'

print(f'School      : {SCHOOL}')
print(f'Raw data    : {RAW_PATH}')

df_raw = pd.read_csv(RAW_PATH)
print(f'Loaded {len(df_raw):,} rows  |  {df_raw["place"].nunique():,} unique places')
print(f"Columns: {list(df_raw.columns)}")
df_raw.head(3)

> 📊 **Output:** You should see the raw geoparsed rows. The `action`, `corrected_name`, `corrected_latlon`, `corrected_place_type`, and `reviewer` columns will be empty. Your team will fill them in in via Google Sheets.

## 2 Setup Google Sheet (One person)

For this section, you will repeat the steps from [Lesson 4.4](../lesson_4_finding_locations/lesson_4_4_preparing_review_sheet.ipynb). Please consult this lesson for more thorough documentation.
**One student** starts this step; the whole team contributes. You will import your school's raw data into Google Sheets, setup data validation, and publish the sheet so this notebook can load the result in section 3 for review.

### 2.1 Import the file

1. Download the file: `data/{SCHOOL}/{SCHOOL}_geoparsed_long.csv` to a local directory
1. Go to [sheets.google.com](https://sheets.google.com) → **Blank spreadsheet**
2. **File → Import → Upload** → select `data/{SCHOOL}/{SCHOOL}_geoparsed_long.csv` from your local machine
3. Choose **Replace spreadsheet**, separator type **Comma**
4. Rename the spreadsheet: `{SCHOOL}_geoparsed_long_cleaned`
6. **Share → Anyone with the link → Editor** → copy the link and post it in your team channel

> 👉 **Note:** *The sheet should be sorted by `place_count` in descending order. Start by batch editing high-frequency places first and work your way down.*

### 2.2 Add data validation

Set up three dropdown validations so the whole team fills in consistent values.

**Column `action`** *(most important)*:

1. Click the `action` column header to select the whole column
2. **Data → Data validation → Add rule**
3. Criteria: **Dropdown** → add three options: `KEEP`, `CORRECT`, `REMOVE`
4. "If data is invalid": **Reject input**
5. Right-click the `action` header cell → **Insert note** → paste: *KEEP = location is correct. CORRECT = right place, wrong details. REMOVE = not a real location or geoparser error.*

**Column `reviewer`**:

1. Select the `reviewer` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add each team member's name

**Column `corrected_place_type`**:

1. Select the `corrected_place_type` column → **Data → Data validation → Add rule**
2. Criteria: **Dropdown** → add: `Country`, `State`, `Region`, `City`, `Neighborhood`, `University`, `Road`, `Building`, `Natural Feature`

### 2.3 Publish the sheet and set `SHEETS_LINK`

Once your team has reviewed all rows:

1. **File → Share → Publish to web**
2. Choose: **Entire document** → **Comma-separated values (.csv)**
3. Click **Publish** → copy the URL
4. Create a *new branch*
5. Scroll back up to the first code cell in this notebook, paste the URL into `SHEETS_LINK`, and re-run that cell
6. Commit, merge, and sync with the rest of the team.





## 3  Review Sheet Google Sheet (Team Effort)

### 3.1 Review the rows

Work through the rows as a team. For each location:

- Read the `sentences` column to understand the context
- Check `place`, `latitude`, `longitude`, and `place_type`
- Set `action` to `KEEP`, `CORRECT`, or `REMOVE`
- If `CORRECT`: fill in only the columns that need changing:
  - `corrected_name` — new place name
  - `corrected_latlon` — paste directly from Google Maps (e.g. `38.433998, -78.872973`)
  - `corrected_place_type` — select from the dropdown
- Select your name in `reviewer`

> 👉 **Note:** *Getting coordinates from Google Maps: Right-click any spot on the map → click the coordinates at the top of the menu → they copy automatically. Paste the full string (`38.433998, -78.872973`) into `corrected_latlon` — the comma is fine, Python will split it on import.*

### 3.2 Validate and Map Data

Run the cell below to validate the data. This makes sure that everything you entered is in a a good place. Fix any validation errors as they might occur.

### 3.3 Review Map

Use the map below to help review your work.

In [ ]:
# Verification map — inspect dots geographically
# Points far outside Virginia / the US East Coast are worth checking
import sys; sys.path.insert(0, "../tests")
from helpers import load_and_validate_review_sheet
import pandas as pd
import plotly.express as px

# ──────────────────────────────────────────────────────────────────────────────
# ⚠️  BEFORE RUNNING THIS CELL:
#   1. Complete Part 2
#   2. File → Share → Publish to web → Entire document → CSV → Publish
#   3. Copy the published URL and paste it below, replacing the placeholder
# ──────────────────────────────────────────────────────────────────────────────

if "PASTE_YOUR_LINK_HERE" in SHEETS_LINK:
    print("❌ Update SHEETS_LINK before running this cell.")
    print(
        "   File → Share → Publish to web → Entire document → CSV → Publish → copy URL."
    )
else:
    df_review = load_and_validate_review_sheet(SHEETS_LINK)

if "df_review" not in dir():
    print("❌ df_review is not loaded yet.")
    print("   Run the cell above first (paste your SHEETS_LINK and run it), then re-run this cell.")
else:
    df_map = df_review[
        df_review["action"].isin(["KEEP", "CORRECT"]) |
        df_review["action"].isna() |
        (df_review["action"] == "")
    ].copy()

    df_map["action"] = df_map["action"].fillna("UNVERIFIED")

    df_map["lat_plot"] = pd.to_numeric(
        df_map["corrected_lat"].where(
            df_map["corrected_lat"].notna() & (df_map["corrected_lat"].astype(str).str.strip() != ""),
            df_map["latitude"]
        ), errors="coerce")

    df_map["lon_plot"] = pd.to_numeric(
        df_map["corrected_lon"].where(
            df_map["corrected_lon"].notna() & (df_map["corrected_lon"].astype(str).str.strip() != ""),
            df_map["longitude"]
        ), errors="coerce")

    df_map["place"] = df_map["corrected_name"].where(
        df_map["corrected_name"].notna() & (df_map["corrected_name"] !=""), df_map["place"])

    df_map["corrected_place_type"] = df_map["corrected_place_type"].where((df_map["action"]=="CORRECT") &
               df_map["corrected_place_type"].notna() & (df_map["corrected_place_type"] !=""), "None")

    df_map = df_map.dropna(subset=["lat_plot", "lon_plot"])

    df_map = (
        df_map.groupby(["place", "action"], sort=False)
        .agg(
            corrected_name=("corrected_name", "first"),
            location_count=("place", "size"),
            lat_plot=("lat_plot", "first"),
            lon_plot=("lon_plot", "first"),
            place_type=("place_type", "first"),
            corrected_place_type=("corrected_place_type", "first"),
            reviewer=("reviewer", "first"),
        )
        .reset_index()
    )

    df_map[["corrected_place_type", "reviewer", "corrected_name"]] = df_map[["corrected_place_type", "reviewer","corrected_name"]].fillna("None")

    fig = px.scatter_map(
        df_map,
        lat="lat_plot", lon="lon_plot",
        size="location_count",
        hover_name="place",
        labels={"lat_plot":"latitude","lon_plot":"longitude"},
        hover_data={
            "corrected_name": True,
            "lat_plot": True,
            "lon_plot": True,
            "place_type": True,
            "corrected_place_type": True,
            "action": True,
            "reviewer": True
        },
        color="action",
        color_discrete_map={
            "KEEP": "#2ca02c",
            "CORRECT": "#ff7f0e",
            "UNVERIFIED": "#aec7e8",
        },
       
        size_max=25,
        map_style="carto-positron",
        center={
            "lat": 37.5,
            "lon": -78.0,
        },
        zoom=4,
        height=500,
        title="Location review map — hover for details"
    )
    fig.update_layout(
        margin={
            "r": 0,
            "t": 50,
            "l": 0,
            "b": 0,
        }
    )
    fig.update_traces(marker=dict(sizemin=3))

    fig.show()

    print("\n💡 Dots far outside Virginia may be geoparser errors.")    

## Apply Corrections and Export (One Person)

This step does two things:
1. Applies all corrections (drops REMOVE rows, overwrites coordinates and place names for CORRECT rows)
2. Strips the review columns and saves the clean result as both `{SCHOOL}_geoparsed_cleaned.csv` (for inspection) and `{SCHOOL}_geoparsed_cleaned.pickle` (for downstream notebooks)

**One student on the team** does these steps after cleaning is complete.

This is the same branch → commit → pull request workflow from [Lesson 1.1](../lesson_1_the_team/lesson_1_1_git_and_pull_requests.ipynb). Refer back to that lesson if you need a refresher.

1. Create a new branch named `location-review` from the status bar
2. Run export cell below
3. Open **Source Control** and stage `{SCHOOL}_geoparsed_cleaned.csv` and `{SCHOOL}_geoparsed_cleaned.pickle`
4. Commit with a message like `review: update {SCHOOL} location data`, then **Commit & Sync**
5. Publish the branch and open a Pull Request from `location-review` → `main`
6. Merge the PR on GitHub, delete the branch, switch back to `main`, and sync

> 👉 **Note:** *Do not commit any other files. If unexpected files appear under Changes, discard them before committing.*

In [ ]:
import sys
sys.path.insert(0, "../tests")
from helpers import apply_review_corrections

if df_review is None:
    print("⛔ Cannot export — fix validation errors first (re-run Part 2).")
    df_cleaned = None
else:
   
    # 1. Apply corrections
    df_corrected = apply_review_corrections(df_review)

    # 2. Drop review columns and save cleaned CSV + pickle
    _review_cols = ["action", "corrected_name", "corrected_latlon", "corrected_lat",
                    "corrected_lon", "corrected_place_type", "place_count"]
    df_cleaned = df_corrected.drop(columns=[c for c in _review_cols if c in df_corrected.columns])
    df_cleaned.to_csv(f'{CLEANED_PATH}.csv', index=False)
    df_cleaned.to_pickle(f'{CLEANED_PATH}.pickle')

    print(f'✅ Cleaned CSV    saved → {CLEANED_PATH}.csv')
    print(f'✅ Cleaned pickle saved → {CLEANED_PATH}.pickle')
    print(f'   {len(df_review):,} rows in  →  {len(df_cleaned):,} rows out  '
          f'({len(df_review) - len(df_cleaned):,} REMOVE rows dropped)')
    print(f'   Unique places: {df_cleaned["place"].nunique():,}')

> 📊 **Output:** The summary shows how many rows were dropped (REMOVE) and how many unique place names remain. 

## 4 Run Sentiment Analysis (One Person)

This step runs every sentence in your cleaned dataset through the RoBERTa sentiment model and adds four score columns: `roberta_neg`, `roberta_neu`, `roberta_pos`, and `roberta_compound`.

> 👉 **Note:** *This step takes **5–10 minutes** depending on how many sentences your school has. A progress bar will appear. Do not interrupt the process. If it fails, re-run from this cell.*

**One student on the team** does these steps after export is complete.

This is the same branch → commit → pull request workflow from [Lesson 1.1](../lesson_1_the_team/lesson_1_1_git_and_pull_requests.ipynb). Refer back to that lesson if you need a refresher.

1. Create a new branch named `sentiment-analysis` from the status bar
2. Run Sentiment Analysis cell below
3. Run Sentiment Analysis Export
4. Open **Source Control** and stage `{SCHOOL}_geoparsed_cleaned_sentiment.csv` and `{SCHOOL}_geoparsed_cleaned_sentiment.pickle`
5. Commit with a message like `review: update {SCHOOL} location data`, then **Commit & Sync**
6. Publish the branch and open a Pull Request from `sentiment-analysis` → `main`
7. Merge the PR on GitHub, delete the branch, switch back to `main`, and sync


### 4.1 Sentiment Analysis (5-10 minutes)

In [ ]:
import sys; sys.path.insert(0, "../lesson_5_sentiment_analysis")
from sentiment_utils import add_sentiment_to_column

if df_cleaned is None:
    print("⛔ No cleaned data available — complete Part 3 first.")
    df_sentiment = None
else:
    print(f'Running sentiment analysis on {len(df_cleaned):,} rows...')
    df_sentiment = add_sentiment_to_column(df_cleaned, "sentences", output_format="compact")
    print(f'\nDone. New columns: {[c for c in df_sentiment.columns if c not in df_cleaned.columns]}')

### 4.1 Sentiment Analysis Export

The following generates the files for export.

In [ ]:
if df_sentiment is None:
    print("⛔ No sentiment data — run the cell above first.")
else:
    df_sentiment.to_pickle(f'{SENTIMENT_PATH}.pickle')
    df_sentiment.to_csv(f'{SENTIMENT_PATH}.csv', index=False)
    print(f'✅ Saved → {SENTIMENT_PATH}.pickle')
    print(f'✅ Saved → {SENTIMENT_PATH}.csv')
    print(f'\nSentiment distribution (compound score):')
    print(f'  Positive (≥ 0.05): {(df_sentiment["roberta_compound"] >= 0.05).sum():,}')
    print(f'  Neutral  (−0.05 – 0.05): '
          f'{((df_sentiment["roberta_compound"] > -0.05) & (df_sentiment["roberta_compound"] < 0.05)).sum():,}')
    print(f'  Negative (≤ −0.05): {(df_sentiment["roberta_compound"] <= -0.05).sum():,}')

> 📊 **Output:** The sentiment distribution shows the overall emotional tone of what people post about locations at your school. Does one category dominate? Is that what you expected?


Everyone in your team should now have the following files:
| File | Why |
|---|---|
| `data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned.csv` | Cleaned data for whitepaper maps (human-readable) |
| `data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned.pickle` | Cleaned data with dtypes preserved (used by downstream notebooks) |
| `data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned_sentiment.csv` | Sentiment-scored data (human-readable) |
| `data/{SCHOOL}/{SCHOOL}_geoparsed_cleaned_sentiment.pickle` | Sentiment-scored data with dtypes preserved (used by all maps) |



---

➡️ **Next:** [Project Part 2 — Whitepaper](project_part_2_whitepaper.ipynb)